# RiskAware-ASR — Fareez follow-up experiments (Tasks 2, 9, 8, and pilot preparation)

**Run in Google Colab (Runtime → Change runtime type → T4 GPU).** This notebook extends the linked public repository. It does not contain the Fareez audio or private transcripts.

**Inputs to supply in Google Drive:** original Fareez audio, the **same** Whisper-base JSON output (`whisper_out/*.json`) for those audio files, and matching `original_text` gold references inside those JSON files. Each JSON should contain `encounter_id`, `original_text`, `whisper_text`, and `words` with `{word, confidence}` entries. Keep the gold references private.

**Outputs:** `whisper_small_fareez/*.json`, base/small aligned labels, cross-model disagreement ranking, matched-budget comparisons, review-burden estimates, and blinded 30-term rating sheets. This notebook does **not** claim any result until it has run on your actual data.

**Study controls:** same audio, fixed Whisper defaults (matching the repository's base transcription call), same normalization and glossary, no audio regeneration, no clinician outcomes invented. Do not commit identifiable clinical audio/transcripts to GitHub. If the previous base run used non-default decoding settings, match those explicitly before interpreting the comparison.

In [ ]:

from pathlib import Path
from collections import Counter
import json

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")

AUDIO_DIR = PROJECT_DIR / "fareez_audio"
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"

print("Audio folder accessible:", AUDIO_DIR.exists())
print("Whisper-base folder accessible:", BASE_DIR.exists())

audio_files = sorted(
    p for p in AUDIO_DIR.rglob("*") if p.is_file()
) if AUDIO_DIR.exists() else []

base_files = sorted(
    p for p in BASE_DIR.rglob("*") if p.is_file()
) if BASE_DIR.exists() else []

print("\n===== FILE COUNTS =====")
print("Audio files:", len(audio_files))
print("Whisper-base files:", len(base_files))

print("\n===== AUDIO FORMATS =====")
print(Counter(p.suffix.lower() for p in audio_files))

print("\n===== TRANSCRIPT FORMATS =====")
print(Counter(p.suffix.lower() for p in base_files))

print("\n===== SAMPLE AUDIO FILENAMES =====")
for f in audio_files[:5]:
    print(f.name)

print("\n===== SAMPLE TRANSCRIPT FILENAMES =====")
for f in base_files[:5]:
    print(f.name)

print("\n===== JSON STRUCTURE =====")
json_files = [
    p for p in base_files if p.suffix.lower() == ".json"
]

if json_files:
    with open(json_files[0], encoding="utf-8") as f:
        sample = json.load(f)

    print("Type:", type(sample).__name__)
    if isinstance(sample, dict):
        print("Keys:", list(sample.keys()))
    elif isinstance(sample, list):
        print("Entries:", len(sample))
        if sample and isinstance(sample[0], dict):
            print("First entry keys:", list(sample[0].keys()))
else:
    print("No JSON files found.")


In [ ]:

import json
from pathlib import Path

AUDIO_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_audio"
)
BASE_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_whisper_out"
)

# Map files using encounter IDs
audio_map = {
    p.stem: p for p in AUDIO_DIR.rglob("*.mp3")
}

json_map = {
    p.stem: p for p in BASE_DIR.rglob("*.json")
}

audio_ids = set(audio_map)
json_ids = set(json_map)

print("===== FILE MATCHING =====")
print("Audio encounters:", len(audio_ids))
print("JSON encounters:", len(json_ids))
print("Matched encounters:", len(audio_ids & json_ids))

print("Audio without JSON:", sorted(audio_ids - json_ids))
print("JSON without audio:", sorted(json_ids - audio_ids))

# Validate content of each matched JSON
missing_references = []
missing_transcripts = []
missing_words = []
id_mismatches = []
invalid_json = []

for encounter_id in sorted(audio_ids & json_ids):
    try:
        with open(json_map[encounter_id], encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            invalid_json.append(encounter_id)
            continue

        if not str(data.get("original_text") or "").strip():
            missing_references.append(encounter_id)

        if not str(data.get("whisper_text") or "").strip():
            missing_transcripts.append(encounter_id)

        words = data.get("words")
        if not isinstance(words, list) or not words:
            missing_words.append(encounter_id)

        if str(data.get("encounter_id")) != encounter_id:
            id_mismatches.append(encounter_id)

    except (OSError, ValueError, TypeError):
        invalid_json.append(encounter_id)

print("\n===== CONTENT VALIDATION =====")
print("Missing references:", len(missing_references))
print("Missing Whisper transcripts:", len(missing_transcripts))
print("Missing word lists:", len(missing_words))
print("Encounter ID mismatches:", len(id_mismatches))
print("Unreadable/invalid JSON:", len(invalid_json))

# Inspect schema of one file
sample_id = sorted(audio_ids & json_ids)[0]

with open(json_map[sample_id], encoding="utf-8") as f:
    sample = json.load(f)

print("\n===== SAMPLE STRUCTURE =====")
print("Encounter:", sample_id)

for key, value in sample.items():
    print(key, ":", type(value).__name__)

words = sample.get("words", [])

if isinstance(words, list) and words:
    print("\nWord entries:", len(words))
    print("First word entry type:", type(words[0]).__name__)

    if isinstance(words[0], dict):
        print("First word entry keys:", list(words[0].keys()))

print("\nValidation completed.")


In [ ]:

!pip -q install -U openai-whisper jiwer
!which ffmpeg

import torch
import whisper
import json
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "GPU not available. Change Runtime to T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))

# Load Whisper-small
model = whisper.load_model("small", device="cuda")

print("Whisper-small loaded successfully!")


In [ ]:

PROJECT_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr"
)

AUDIO_DIR = PROJECT_DIR / "fareez_audio"
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"

SMALL_DIR = PROJECT_DIR / "fareez_whisper_small"
SMALL_DIR.mkdir(parents=True, exist_ok=True)

PILOT_IDS = ["CAR0001", "CAR0002", "CAR0003"]

for encounter_id in PILOT_IDS:

    audio_path = AUDIO_DIR / f"{encounter_id}.mp3"
    base_path = BASE_DIR / f"{encounter_id}.json"
    output_path = SMALL_DIR / f"{encounter_id}.json"

    assert audio_path.exists()
    assert base_path.exists()

    print(f"\nProcessing {encounter_id}...")

    # Load reference transcript
    with open(base_path, "r", encoding="utf-8") as f:
        base_data = json.load(f)

    # Transcribe using Whisper-small
    result = model.transcribe(
        str(audio_path),
        task="transcribe",
        language="en",
        word_timestamps=True,
        fp16=True,
        verbose=False
    )

    # Extract word-level information
    words = []

    for segment in result["segments"]:
        for w in segment.get("words", []):
            words.append({
                "word": w["word"].strip(),
                "confidence": float(w["probability"])
            })

    # Preserve the original JSON structure
    small_data = {
        "encounter_id": encounter_id,
        "original_text": base_data["original_text"],
        "whisper_text": result["text"].strip(),
        "words": words
    }

    # Save to Google Drive
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            small_data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("Saved:", output_path.name)
    print("Words extracted:", len(words))
    print("Reference characters:",
          len(base_data["original_text"]))
    print("Transcript characters:",
          len(result["text"]))

print("\nPilot transcription completed!")


In [ ]:

print("===== WHISPER-SMALL PILOT =====")

for encounter_id in PILOT_IDS:

    path = SMALL_DIR / f"{encounter_id}.json"

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    confidences = [
        w["confidence"] for w in data["words"]
    ]

    print(f"\nEncounter: {encounter_id}")
    print("JSON keys:", list(data.keys()))
    print("Number of words:", len(data["words"]))
    print("Confidence range:",
          round(min(confidences), 4),
          "to",
          round(max(confidences), 4))
    print("Saved successfully:", path.exists())


In [ ]:

import json
import re
import csv
import jiwer
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
SMALL_DIR = PROJECT_DIR / "fareez_whisper_small"
RESULTS_DIR = PROJECT_DIR / "fareez_followup_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PILOT_IDS = ["CAR0001", "CAR0002", "CAR0003"]

# Same normalization as the project's evaluation script
def normalize(word):
    stripped = re.sub(r"[^a-z0-9']", "", word.lower())
    if stripped == "okay":
        return "ok"
    return stripped

def tokenize(text):
    return [
        w for w in text.split()
        if re.search(r"[a-zA-Z0-9]", w)
    ]

def evaluate(data):
    reference_words = [
        normalize(w)
        for w in tokenize(data["original_text"])
    ]

    hypothesis_words = [
        normalize(w["word"].strip()) or "blank"
        for w in data["words"]
    ]

    output = jiwer.process_words(
        " ".join(reference_words) or "blank",
        " ".join(hypothesis_words) or "blank"
    )

    return {
        "wer": output.wer,
        "reference_words": len(reference_words),
        "substitutions": output.substitutions,
        "deletions": output.deletions,
        "insertions": output.insertions
    }

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

rows = []
totals = {
    "base": {"errors": 0, "words": 0},
    "small": {"errors": 0, "words": 0}
}

print("===== PILOT WER COMPARISON =====")

for eid in PILOT_IDS:
    base = load_json(BASE_DIR / f"{eid}.json")
    small = load_json(SMALL_DIR / f"{eid}.json")

    assert base["original_text"] == small["original_text"], (
        f"Reference mismatch: {eid}"
    )

    b = evaluate(base)
    s = evaluate(small)

    rows.append({
        "encounter_id": eid,
        "base_wer": b["wer"],
        "small_wer": s["wer"],
        "improvement_pp": 100 * (b["wer"] - s["wer"]),
        "reference_words": b["reference_words"]
    })

    for name, result in [("base", b), ("small", s)]:
        errors = (
            result["substitutions"]
            + result["deletions"]
            + result["insertions"]
        )
        totals[name]["errors"] += errors
        totals[name]["words"] += result["reference_words"]

    print(f"\n{eid}")
    print(f"Whisper-base WER:  {b['wer']:.2%}")
    print(f"Whisper-small WER: {s['wer']:.2%}")
    print(f"Difference: {100*(b['wer']-s['wer']):+.2f} pp")

print("\n===== POOLED PILOT RESULTS =====")
for name in ["base", "small"]:
    t = totals[name]
    print(f"Whisper-{name}: {100*t['errors']/t['words']:.2f}%")

# Save results
output_file = RESULTS_DIR / "pilot_base_vs_small_wer.csv"

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("\nSaved:", output_file)


In [ ]:

total_errors = 0
total_reference_words = 0

for path in sorted(BASE_DIR.glob("*.json")):
    data = load_json(path)
    r = evaluate(data)

    total_errors += (
        r["substitutions"]
        + r["deletions"]
        + r["insertions"]
    )
    total_reference_words += r["reference_words"]

pooled_wer = total_errors / total_reference_words

print("===== WHISPER-BASE BASELINE AUDIT =====")
print("Encounters:", len(list(BASE_DIR.glob("*.json"))))
print("Total reference words:", total_reference_words)
print("Total errors:", total_errors)
print(f"Pooled WER: {pooled_wer:.2%}")
print("Previously reported WER: 13.92%")

if abs(pooled_wer - 0.1392) < 0.0001:
    print("Baseline reproduced within rounding tolerance.")
else:
    print("Baseline differs; check data and evaluation settings.")


In [ ]:

import json
import whisper
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")
AUDIO_DIR = PROJECT_DIR / "fareez_audio"
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
SMALL_DIR = PROJECT_DIR / "fareez_whisper_small"

AUTO_DIR = PROJECT_DIR / "fareez_small_auto_pilot"
AUTO_DIR.mkdir(parents=True, exist_ok=True)

PILOT_IDS = ["CAR0001", "CAR0002", "CAR0003"]

# Reuse the already-loaded Whisper-small model
for eid in PILOT_IDS:

    audio_path = AUDIO_DIR / f"{eid}.mp3"

    result = model.transcribe(
        str(audio_path),
        word_timestamps=True,
        fp16=True,
        verbose=False
    )

    with open(BASE_DIR / f"{eid}.json", encoding="utf-8") as f:
        base_data = json.load(f)

    auto_data = {
        "encounter_id": eid,
        "original_text": base_data["original_text"],
        "whisper_text": result["text"],
        "words": [
            {
                "word": w["word"],
                "confidence": float(w["probability"])
            }
            for seg in result["segments"]
            for w in seg.get("words", [])
        ]
    }

    with open(AUTO_DIR / f"{eid}.json", "w", encoding="utf-8") as f:
        json.dump(auto_data, f, ensure_ascii=False, indent=2)

    # Reuse the evaluate() function from Step 5
    base_wer = evaluate(base_data)["wer"]

    with open(SMALL_DIR / f"{eid}.json", encoding="utf-8") as f:
        fixed_data = json.load(f)

    fixed_wer = evaluate(fixed_data)["wer"]
    auto_wer = evaluate(auto_data)["wer"]

    print(f"\n{eid}")
    print("Detected language:", result["language"])
    print(f"Base WER:        {base_wer:.2%}")
    print(f"Small English:   {fixed_wer:.2%}")
    print(f"Small automatic: {auto_wer:.2%}")

print("\nDecoding comparison complete.")


In [ ]:

import json
import shutil
import time
import torch
import whisper
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")

AUDIO_DIR = PROJECT_DIR / "fareez_audio"
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
PILOT_DIR = PROJECT_DIR / "fareez_small_auto_pilot"

OUTPUT_DIR = PROJECT_DIR / "fareez_whisper_small_auto"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load model (cached after the first download)
assert torch.cuda.is_available(), "Please enable a GPU runtime."
model = whisper.load_model("small", device="cuda")

audio_map = {p.stem: p for p in AUDIO_DIR.glob("*.mp3")}
base_map = {p.stem: p for p in BASE_DIR.glob("*.json")}

assert set(audio_map) == set(base_map), "Audio/JSON IDs differ."
encounter_ids = sorted(audio_map)

print("Total encounters:", len(encounter_ids))

# Record experiment configuration
config = {
    "model": "Whisper-small",
    "language": "automatic_detection",
    "task": "transcribe",
    "word_timestamps": True,
    "fp16": True,
    "torch_version": torch.__version__,
    "started_utc": datetime.now(timezone.utc).isoformat()
}

config_path = OUTPUT_DIR / "experiment_config.json"
if not config_path.exists():
    config_path.write_text(
        json.dumps(config, indent=2), encoding="utf-8"
    )

def valid_output(path, eid, reference):
    try:
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        return (
            isinstance(d, dict)
            and d.get("encounter_id") == eid
            and d.get("original_text") == reference
            and isinstance(d.get("whisper_text"), str)
            and bool(d["whisper_text"].strip())
            and isinstance(d.get("words"), list)
            and len(d["words"]) > 0
            and all(
                isinstance(w, dict)
                and "word" in w
                and "confidence" in w
                for w in d["words"]
            )
        )
    except (OSError, ValueError, TypeError, KeyError):
        return False

# Reuse the automatic-language pilot files
for eid in ["CAR0001", "CAR0002", "CAR0003"]:
    source = PILOT_DIR / f"{eid}.json"
    target = OUTPUT_DIR / f"{eid}.json"

    with open(base_map[eid], encoding="utf-8") as f:
        reference = json.load(f)["original_text"]

    if not target.exists() and valid_output(source, eid, reference):
        shutil.copy2(source, target)
        print("Reused pilot:", eid)

completed = 0
skipped = 0
failed = []

for index, eid in enumerate(encounter_ids, start=1):

    output_path = OUTPUT_DIR / f"{eid}.json"

    with open(base_map[eid], encoding="utf-8") as f:
        base_data = json.load(f)

    reference = base_data["original_text"]

    if valid_output(output_path, eid, reference):
        skipped += 1
        continue

    try:
        start = time.perf_counter()

        result = model.transcribe(
            str(audio_map[eid]),
            task="transcribe",
            language=None,
            word_timestamps=True,
            fp16=True,
            verbose=False
        )

        words = [
            {
                "word": w["word"],
                "confidence": float(w["probability"])
            }
            for seg in result["segments"]
            for w in seg.get("words", [])
        ]

        if not words:
            raise ValueError("No words extracted")

        data = {
            "encounter_id": eid,
            "original_text": reference,
            "whisper_text": result["text"].strip(),
            "words": words
        }

        # Save to a temporary file first
        temp_path = OUTPUT_DIR / f"{eid}.json.partial"

        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        temp_path.replace(output_path)

        completed += 1
        elapsed = time.perf_counter() - start

        print(
            f"[{index}/{len(encounter_ids)}] {eid} | "
            f"{len(words)} words | {elapsed:.1f}s | Saved",
            flush=True
        )

    except Exception as e:
        failed.append((eid, str(e)))
        print(f"FAILED: {eid}: {e}", flush=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\n========== BATCH SUMMARY ==========")
print("Newly processed:", completed)
print("Previously completed:", skipped)
print("Failures:", len(failed))
print("Total saved JSON files:",
      len(list(OUTPUT_DIR.glob("CAR*.json"))))

if failed:
    print("Failed encounters:", failed)


In [ ]:

import json

valid_ids = []
invalid_ids = []

for eid in encounter_ids:
    output = OUTPUT_DIR / f"{eid}.json"

    with open(base_map[eid], encoding="utf-8") as f:
        reference = json.load(f)["original_text"]

    if valid_output(output, eid, reference):
        valid_ids.append(eid)
    else:
        invalid_ids.append(eid)

print("===== FINAL VALIDATION =====")
print("Expected encounters:", len(encounter_ids))
print("Valid Whisper-small outputs:", len(valid_ids))
print("Missing or invalid outputs:", len(invalid_ids))
print("IDs requiring attention:", invalid_ids)

if not invalid_ids:
    print("SUCCESS: Full Whisper-small dataset complete!")
else:
    print("Rerun Step 7.1 to process remaining encounters.")


In [ ]:

assert len(valid_ids) == 272, "Complete the batch first."

base_errors = 0
small_errors = 0
total_words = 0

for eid in encounter_ids:

    with open(BASE_DIR / f"{eid}.json", encoding="utf-8") as f:
        base = json.load(f)

    with open(OUTPUT_DIR / f"{eid}.json", encoding="utf-8") as f:
        small = json.load(f)

    assert base["original_text"] == small["original_text"]

    b = evaluate(base)
    s = evaluate(small)

    base_errors += (
        b["substitutions"] + b["deletions"] + b["insertions"]
    )
    small_errors += (
        s["substitutions"] + s["deletions"] + s["insertions"]
    )
    total_words += b["reference_words"]

base_wer = base_errors / total_words
small_wer = small_errors / total_words

print("===== FULL DATASET COMPARISON =====")
print("Encounters:", len(encounter_ids))
print("Reference words:", total_words)
print(f"Whisper-base WER:  {base_wer:.2%}")
print(f"Whisper-small WER: {small_wer:.2%}")
print(f"WER reduction: {100*(base_wer-small_wer):+.2f} pp")
print(
    f"Relative WER reduction: "
    f"{100*(base_wer-small_wer)/base_wer:.2f}%"
)


In [ ]:

import sys
import subprocess
import importlib.util
from pathlib import Path
import pandas as pd

REPO_DIR = Path("/content/Risk-aware-ASR")

if not (REPO_DIR / ".git").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/mih1ka/Risk-aware-ASR.git",
        str(REPO_DIR)
    ], check=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

print("Repository commit:", commit)

# Load the existing alignment implementation
script_path = REPO_DIR / "src/eval/align_and_label.py"

spec = importlib.util.spec_from_file_location(
    "riskaware_alignment", script_path
)
alignment_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(alignment_module)

# Load the tiered glossary
glossary_path = (
    REPO_DIR / "data/glossary/tiered_glossary.csv"
)

glossary_df = pd.read_csv(glossary_path)

glossary = dict(zip(
    glossary_df["term"].str.lower(),
    glossary_df["tier"]
))

print("\n===== GLOSSARY =====")
print("Total terms:", len(glossary_df))
print(glossary_df["tier"].value_counts().sort_index())

assert len(glossary_df) == 190, (
    "Glossary differs from the reported 190-term version."
)

print("\nGlossary loaded successfully!")


In [ ]:

# ============================================================
# STEP 8.2 — CORRECTED WORD-LEVEL ALIGNMENT
# Whisper-base vs Whisper-small (272 Fareez encounters)
# ============================================================

from pathlib import Path

# 1. Define directories
PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")

BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
SMALL_DIR = PROJECT_DIR / "fareez_whisper_small_auto"
RESULTS_DIR = PROJECT_DIR / "fareez_followup_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Collect all Whisper-base encounter files
base_files = sorted(BASE_DIR.glob("*.json"))

# 3. Match Whisper-small files using identical filenames
# This excludes experiment_config.json automatically
small_files = [
    SMALL_DIR / file.name
    for file in base_files
]

# 4. Verify all files exist
missing_files = [
    p.name for p in small_files
    if not p.is_file()
]

print("===== FILE MATCHING =====")
print("Whisper-base files:", len(base_files))
print("Whisper-small files:", len(small_files))
print("Missing files:", missing_files)

assert len(base_files) == 272, (
    "Expected 272 Whisper-base files."
)

assert len(small_files) == 272, (
    "Expected 272 Whisper-small files."
)

assert not missing_files, (
    f"Missing Whisper-small files: {missing_files}"
)

print("\nAll 272 encounters matched successfully!")

# 5. Perform word-level alignment
# Uses alignment_module and glossary loaded in Step 8.1

print("\nAligning Whisper-base...")

base_labels, base_enc_wer = (
    alignment_module.align_files(
        base_files, glossary
    )
)

print("Whisper-base alignment completed!")

print("\nAligning Whisper-small...")

small_labels, small_enc_wer = (
    alignment_module.align_files(
        small_files, glossary
    )
)

print("Whisper-small alignment completed!")

# 6. Validate aligned encounters
base_ids = set(base_labels["encounter_id"])
small_ids = set(small_labels["encounter_id"])

assert base_ids == small_ids, (
    "Encounter IDs do not match after alignment."
)

assert len(base_ids) == 272, (
    "Expected 272 aligned encounters."
)

# 7. Save word-level labels
base_labels.to_csv(
    RESULTS_DIR / "fareez_base_word_labels.csv",
    index=False
)

small_labels.to_csv(
    RESULTS_DIR / "fareez_small_word_labels.csv",
    index=False
)

# 8. Save encounter-level WER
base_enc_wer.to_csv(
    RESULTS_DIR / "fareez_base_encounter_wer.csv",
    index=False
)

small_enc_wer.to_csv(
    RESULTS_DIR / "fareez_small_encounter_wer.csv",
    index=False
)

# 9. Display summary
print("\n========== ALIGNMENT SUMMARY ==========")

print("Total encounters:", len(base_ids))
print("Whisper-base word rows:", len(base_labels))
print("Whisper-small word rows:", len(small_labels))

print("\nMean encounter-level WER:")
print(f"Whisper-base:  {base_enc_wer['wer'].mean():.2%}")
print(f"Whisper-small: {small_enc_wer['wer'].mean():.2%}")

print("\n========== SAVED OUTPUT FILES ==========")
for name in [
    "fareez_base_word_labels.csv",
    "fareez_small_word_labels.csv",
    "fareez_base_encounter_wer.csv",
    "fareez_small_encounter_wer.csv"
]:
    print(name)

print("\nSTEP 8.2 COMPLETED SUCCESSFULLY!")


In [ ]:

# Use alignment_module and glossary from Step 8.1

print("Aligning Whisper-base...")
base_labels, base_enc_wer = alignment_module.align_files(
    base_files, glossary
)

print("Aligning Whisper-small...")
small_labels, small_enc_wer = alignment_module.align_files(
    small_files, glossary
)

# Save word-level labels
base_labels.to_csv(
    RESULTS_DIR / "fareez_base_word_labels.csv",
    index=False
)

small_labels.to_csv(
    RESULTS_DIR / "fareez_small_word_labels.csv",
    index=False
)

# Save encounter-level WER
base_enc_wer.to_csv(
    RESULTS_DIR / "fareez_base_encounter_wer.csv",
    index=False
)

small_enc_wer.to_csv(
    RESULTS_DIR / "fareez_small_encounter_wer.csv",
    index=False
)

print("\n===== ALIGNMENT RESULTS =====")
print("Base word-level rows:", len(base_labels))
print("Small word-level rows:", len(small_labels))

print("Base encounters:",
      base_labels["encounter_id"].nunique())
print("Small encounters:",
      small_labels["encounter_id"].nunique())

print("\nFiles saved successfully!")


In [ ]:

import pandas as pd

def tier_summary(labels):
    gold = labels[labels["gold_word"].notna()].copy()

    summary = gold.groupby("tier")["is_error"].agg(
        errors="sum",
        reference_words="count",
        error_rate="mean"
    )

    return summary.reindex([1, 2, 3])

base_tiers = tier_summary(base_labels)
small_tiers = tier_summary(small_labels)

# Both models must have identical reference denominators
assert base_tiers["reference_words"].equals(
    small_tiers["reference_words"]
), "Reference word counts differ!"

comparison = pd.DataFrame({
    "base_wer": base_tiers["error_rate"],
    "small_wer": small_tiers["error_rate"],
    "reference_words": base_tiers["reference_words"],
    "base_errors": base_tiers["errors"],
    "small_errors": small_tiers["errors"]
})

comparison["improvement_pp"] = (
    comparison["base_wer"] - comparison["small_wer"]
) * 100

print("===== TIER-WISE ERROR COMPARISON =====")

for tier, row in comparison.iterrows():
    print(f"\nTIER {tier}")
    print("Reference words:", int(row["reference_words"]))
    print(f"Whisper-base:  {row['base_wer']:.2%}")
    print(f"Whisper-small: {row['small_wer']:.2%}")
    print(f"Improvement:   {row['improvement_pp']:+.2f} pp")

comparison.to_csv(
    RESULTS_DIR / "base_vs_small_tier_comparison.csv"
)

print("\n===== BASELINE REPRODUCTION =====")
expected = {1: 0.1024, 2: 0.2433, 3: 0.5504}

for tier, reported in expected.items():
    observed = float(base_tiers.loc[tier, "error_rate"])

    status = (
        "MATCH"
        if abs(observed - reported) < 0.0001
        else "CHECK REQUIRED"
    )

    print(
        f"Tier {tier}: {observed:.2%} "
        f"(reported {reported:.2%}) — {status}"
    )

print("\nTotal reference words:",
      int(base_tiers["reference_words"].sum()))
print("Saved tier comparison!")


In [ ]:

# ============================================================
# STEP 8.5 — PAIRED ENCOUNTER-LEVEL BOOTSTRAP
# Whisper-base vs Whisper-small
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

base = pd.read_csv(
    RESULTS_DIR / "fareez_base_word_labels.csv"
)
small = pd.read_csv(
    RESULTS_DIR / "fareez_small_word_labels.csv"
)

def prepare_counts(df):
    # Match the reference-token definition from Step 8.4
    gold = df[df["gold_word"].notna()].copy()

    gold["tier"] = pd.to_numeric(gold["tier"])
    gold["is_error"] = pd.to_numeric(gold["is_error"])

    assert gold["tier"].isin([1, 2, 3]).all()
    assert gold["is_error"].isin([0, 1]).all()

    return gold.groupby(
        ["encounter_id", "tier"]
    )["is_error"].agg(
        errors="sum",
        words="count"
    )

base_counts = prepare_counts(base)
small_counts = prepare_counts(small)

encounters = sorted(
    set(base["encounter_id"]) &
    set(small["encounter_id"])
)

assert len(encounters) == 272

index = pd.MultiIndex.from_product(
    [encounters, [1, 2, 3]],
    names=["encounter_id", "tier"]
)

b = base_counts.reindex(index, fill_value=0)
s = small_counts.reindex(index, fill_value=0)

# Identical reference denominators for every encounter/tier
assert np.array_equal(
    b["words"].to_numpy(),
    s["words"].to_numpy()
), "Reference denominators differ."

# Shape: 272 encounters x 3 tiers
n = b["words"].to_numpy().reshape(272, 3)
be = b["errors"].to_numpy().reshape(272, 3)
se = s["errors"].to_numpy().reshape(272, 3)

rng = np.random.default_rng(42)
N_BOOT = 5000

# Sample whole encounters, preserving model pairing
samples = rng.integers(
    0, len(encounters),
    size=(N_BOOT, len(encounters))
)

results = []

for j, tier in enumerate([1, 2, 3]):
    total_n = n[:, j].sum()

    base_rate = be[:, j].sum() / total_n
    small_rate = se[:, j].sum() / total_n

    # Each resample uses identical encounters for both models
    boot_n = n[samples, j].sum(axis=1)
    boot_b = be[samples, j].sum(axis=1)
    boot_s = se[samples, j].sum(axis=1)

    valid = boot_n > 0

    # Positive values represent lower error under small
    boot_difference = (
        boot_b[valid] / boot_n[valid]
        - boot_s[valid] / boot_n[valid]
    ) * 100

    ci_low, ci_high = np.percentile(
        boot_difference, [2.5, 97.5]
    )

    reduction_pp = (base_rate - small_rate) * 100

    results.append({
        "tier": tier,
        "reference_words": int(total_n),
        "base_errors": int(be[:, j].sum()),
        "small_errors": int(se[:, j].sum()),
        "base_error_rate_pct": base_rate * 100,
        "small_error_rate_pct": small_rate * 100,
        "reduction_pp": reduction_pp,
        "ci_lower_pp": ci_low,
        "ci_upper_pp": ci_high,
        "relative_reduction_pct": (
            100 * (base_rate - small_rate) / base_rate
        )
    })

summary = pd.DataFrame(results)

print("===== BOOTSTRAP RESULTS =====")
print(summary.round(3).to_string(index=False))

print("\n===== INTERPRETATION =====")

for _, row in summary.iterrows():
    tier = int(row["tier"])

    print(f"\nTier {tier}")
    print(f"Error-rate reduction: {row['reduction_pp']:.2f} pp")
    print(
        f"95% CI: [{row['ci_lower_pp']:.2f}, "
        f"{row['ci_upper_pp']:.2f}] pp"
    )

    if row["ci_lower_pp"] > 0:
        print("CI excludes zero; supports a positive improvement.")
    else:
        print("CI includes zero; improvement remains uncertain.")

output_path = RESULTS_DIR / "tier_bootstrap_95ci.csv"
summary.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("STEP 8.5 COMPLETE!")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

base_path = RESULTS_DIR / "fareez_base_word_labels.csv"
small_path = RESULTS_DIR / "fareez_small_word_labels.csv"

# Preserve literal words such as "NA" and "None"
base = pd.read_csv(base_path, keep_default_na=False)
small = pd.read_csv(small_path, keep_default_na=False)

def prepare(df):
    gold = df[
        df["gold_word"].astype(str).str.strip().ne("")
    ].copy()

    gold["tier"] = pd.to_numeric(gold["tier"]).astype(int)
    gold["is_error"] = pd.to_numeric(
        gold["is_error"]
    ).astype(int)

    return gold.groupby(
        ["encounter_id", "tier"]
    )["is_error"].agg(
        errors="sum",
        words="count"
    )

b_counts = prepare(base)
s_counts = prepare(small)

ids = sorted(set(base.encounter_id) & set(small.encounter_id))

idx = pd.MultiIndex.from_product(
    [ids, [1, 2, 3]],
    names=["encounter_id", "tier"]
)

b = b_counts.reindex(idx, fill_value=0)
s = s_counts.reindex(idx, fill_value=0)

assert len(ids) == 272
assert np.array_equal(b["words"].values, s["words"].values)

n = b["words"].to_numpy().reshape(272, 3)
be = b["errors"].to_numpy().reshape(272, 3)
se = s["errors"].to_numpy().reshape(272, 3)

print("===== DENOMINATOR AUDIT =====")
print("Reference words:", n.sum())
print("Tier counts:", n.sum(axis=0))

assert n.sum() == 357503, (
    "Reference count still differs; investigate before proceeding."
)

rng = np.random.default_rng(42)
samples = rng.integers(0, 272, size=(5000, 272))

results = []

for j, tier in enumerate([1, 2, 3]):
    denominator = n[:, j].sum()

    base_rate = be[:, j].sum() / denominator
    small_rate = se[:, j].sum() / denominator

    boot_n = n[samples, j].sum(axis=1)
    boot_b = be[samples, j].sum(axis=1)
    boot_s = se[samples, j].sum(axis=1)

    valid = boot_n > 0

    difference = 100 * (
        boot_b[valid] / boot_n[valid]
        - boot_s[valid] / boot_n[valid]
    )

    ci = np.percentile(difference, [2.5, 97.5])

    results.append({
        "tier": tier,
        "reference_words": int(denominator),
        "base_errors": int(be[:, j].sum()),
        "small_errors": int(se[:, j].sum()),
        "base_error_rate_pct": 100 * base_rate,
        "small_error_rate_pct": 100 * small_rate,
        "reduction_pp": 100 * (base_rate - small_rate),
        "ci_lower_pp": ci[0],
        "ci_upper_pp": ci[1],
        "relative_reduction_pct":
            100 * (base_rate - small_rate) / base_rate
    })

summary = pd.DataFrame(results)

print("\n===== CORRECTED BOOTSTRAP RESULTS =====")
print(summary.round(3).to_string(index=False))

summary.to_csv(
    RESULTS_DIR / "tier_bootstrap_95ci.csv",
    index=False
)

print("\nCorrected results saved successfully!")


In [ ]:

# ============================================================
# TASK 9 — STEP 9.1
# Cross-model word-level disagreement
# Whisper-base vs Whisper-small
# ============================================================

import json
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import jiwer

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")
BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
SMALL_DIR = PROJECT_DIR / "fareez_whisper_small_auto"
RESULTS_DIR = PROJECT_DIR / "fareez_followup_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the repository's original normalization functions
script_path = Path(
    "/content/Risk-aware-ASR/src/eval/align_and_label.py"
)

spec = importlib.util.spec_from_file_location(
    "riskaware_alignment", script_path
)
alignment_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(alignment_module)

normalize = alignment_module.normalize
tokenize = alignment_module.tokenize

# Load the same 190-term glossary
glossary_path = Path(
    "/content/Risk-aware-ASR/data/glossary/tiered_glossary.csv"
)

glossary_df = pd.read_csv(
    glossary_path, keep_default_na=False
)

glossary = dict(zip(
    glossary_df["term"].str.lower(),
    glossary_df["tier"].astype(int)
))

base_files = sorted(BASE_DIR.glob("*.json"))

assert len(base_files) == 272
assert all(
    (SMALL_DIR / p.name).is_file()
    for p in base_files
)

def get_alignment(left, right):
    result = jiwer.process_words(
        " ".join(left) or "blank",
        " ".join(right) or "blank"
    )
    return result.alignments[0]

rows = []
tier_words = {1: 0, 2: 0, 3: 0}
tier_errors = {1: 0, 2: 0, 3: 0}

total_small_insertions = 0
unscorable_critical = 0

for path in base_files:

    eid = path.stem

    with open(path, encoding="utf-8") as f:
        base = json.load(f)

    with open(SMALL_DIR / path.name, encoding="utf-8") as f:
        small = json.load(f)

    assert base["original_text"] == small["original_text"]
    assert base["encounter_id"] == small["encounter_id"]

    gold_raw = tokenize(base["original_text"])
    gold_norm = [normalize(w) for w in gold_raw]

    bw = base["words"]
    sw = small["words"]

    base_norm = [
        normalize(w["word"].strip()) or "blank"
        for w in bw
    ]

    small_norm = [
        normalize(w["word"].strip()) or "blank"
        for w in sw
    ]

    # --------------------------------------------------------
    # A. Reference-based labels — evaluation ONLY
    # --------------------------------------------------------
    gold_error = [0] * len(gold_norm)
    base_to_gold = [None] * len(bw)

    for chunk in get_alignment(gold_norm, base_norm):

        # Mark substituted/deleted reference words as errors
        if chunk.type != "equal":
            for ri in range(
                chunk.ref_start_idx, chunk.ref_end_idx
            ):
                gold_error[ri] = 1

        # Map base output tokens to reference positions
        if chunk.type in ("equal", "substitute"):

            for offset, bi in enumerate(
                range(chunk.hyp_start_idx, chunk.hyp_end_idx)
            ):
                ri = chunk.ref_start_idx + offset

                if ri < chunk.ref_end_idx:
                    base_to_gold[bi] = ri

    for ri, word in enumerate(gold_norm):
        tier = int(glossary.get(word, 1))
        tier_words[tier] += 1
        tier_errors[tier] += gold_error[ri]

    mapped_gold = {
        ri for ri in base_to_gold if ri is not None
    }

    unscorable_critical += sum(
        gold_error[ri] == 1
        and glossary.get(gold_norm[ri], 1) >= 2
        and ri not in mapped_gold
        for ri in range(len(gold_norm))
    )

    # --------------------------------------------------------
    # B. Base vs small alignment — reference-free
    # --------------------------------------------------------
    disagreement = [0] * len(bw)
    small_aligned = [""] * len(bw)

    for chunk in get_alignment(base_norm, small_norm):

        if chunk.type == "equal":

            for offset, bi in enumerate(
                range(chunk.ref_start_idx, chunk.ref_end_idx)
            ):
                si = chunk.hyp_start_idx + offset
                small_aligned[bi] = sw[si]["word"].strip()

        elif chunk.type in ("substitute", "delete"):

            for offset, bi in enumerate(
                range(chunk.ref_start_idx, chunk.ref_end_idx)
            ):
                disagreement[bi] = 1

                si = chunk.hyp_start_idx + offset

                if (
                    chunk.type == "substitute"
                    and si < chunk.hyp_end_idx
                ):
                    small_aligned[bi] = sw[si]["word"].strip()

        elif chunk.type == "insert":
            # Small-only words have no base output token
            total_small_insertions += (
                chunk.hyp_end_idx - chunk.hyp_start_idx
            )

    # --------------------------------------------------------
    # C. Save one row per Whisper-base output word
    # --------------------------------------------------------
    for bi, word in enumerate(bw):

        ri = base_to_gold[bi]

        if ri is None:
            gold_word = ""
            tier = 0
            is_error = 1  # Base inserted a word
            is_critical = 0

        else:
            gold_word = gold_raw[ri]
            tier = int(glossary.get(gold_norm[ri], 1))
            is_error = gold_error[ri]
            is_critical = int(tier >= 2 and is_error == 1)

        rows.append({
            "encounter_id": eid,
            "base_word_index": bi,
            "base_word": word["word"].strip(),
            "small_aligned_word": small_aligned[bi],
            "base_confidence": float(word["confidence"]),
            "disagreement": disagreement[bi],
            "gold_word": gold_word,
            "gold_tier": tier,
            "base_is_error": is_error,
            "base_is_critical_error": is_critical
        })

# ------------------------------------------------------------
# D. Integrity checks against your verified results
# ------------------------------------------------------------
assert sum(tier_words.values()) == 357503, (
    "Reference-token count mismatch."
)

assert tier_words == {
    1: 356873, 2: 263, 3: 367
}, "Glossary tier counts differ."

assert tier_errors == {
    1: 36538, 2: 64, 3: 202
}, "Base error labels differ from verified baseline."

df = pd.DataFrame(rows)

assert df["encounter_id"].nunique() == 272
assert df["disagreement"].isin([0, 1]).all()

output_path = (
    RESULTS_DIR / "fareez_cross_model_disagreement.csv"
)

df.to_csv(output_path, index=False)

# ------------------------------------------------------------
# E. Summary
# ------------------------------------------------------------
total = len(df)
disagreed = int(df["disagreement"].sum())

critical_total = tier_errors[2] + tier_errors[3]
critical_caught = int(
    df.loc[
        df["disagreement"] == 1,
        "base_is_critical_error"
    ].sum()
)

print("===== CROSS-MODEL DISAGREEMENT =====")
print("Encounters:", df["encounter_id"].nunique())
print("Base output words:", total)
print("Disagreed base words:", disagreed)
print(f"Disagreement rate: {disagreed/total:.2%}")

print("\n===== CRITICAL ERRORS =====")
print("Total Tier 2/3 base errors:", critical_total)
print("Critical errors at disagreed positions:", critical_caught)
print("Critical errors without a base-token match:",
      unscorable_critical)

print("\nSmall-only insertions during alignment:",
      total_small_insertions)

print("\nSaved:", output_path)
print("STEP 9.1 COMPLETE!")


In [ ]:

# ============================================================
# TASK 9 — STEP 9.2
# Fixed-budget cross-model disagreement evaluation
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

df = pd.read_csv(
    RESULTS_DIR / "fareez_cross_model_disagreement.csv",
    keep_default_na=False
)

dis = pd.to_numeric(
    df["disagreement"]
).to_numpy(dtype=int)

conf = pd.to_numeric(
    df["base_confidence"]
).to_numpy(dtype=float)

critical = pd.to_numeric(
    df["base_is_critical_error"]
).to_numpy(dtype=int)

assert np.isfinite(conf).all()
assert np.all((conf >= 0) & (conf <= 1))
assert np.isin(dis, [0, 1]).all()
assert np.isin(critical, [0, 1]).all()

N = len(df)
N_CRITICAL_ALL = 266
N_CRITICAL_SCORABLE = int(critical.sum())

assert N == 352479
assert dis.sum() == 28896
assert N_CRITICAL_SCORABLE == 245
assert critical[dis == 1].sum() == 207

print("===== DATASET AUDIT =====")
print("Base output words:", N)
print("Disagreement flags:", dis.sum())
print("Critical errors in scored rows:",
      N_CRITICAL_SCORABLE)
print("All reference critical errors:",
      N_CRITICAL_ALL)

# Lowest confidence first
idx = np.arange(N)
confidence_order = np.lexsort((idx, conf))

# Primary ranking: disagreement first
# Secondary ranking: lowest confidence first
hybrid_order = np.lexsort((idx, conf, -dis))

def calculate_metrics(order, n_flags, method, budget):
    selected = order[:n_flags]

    caught = int(critical[selected].sum())

    return {
        "method": method,
        "budget_pct": budget * 100,
        "flagged_words": n_flags,
        "critical_caught": caught,
        "recall_all_pct":
            100 * caught / N_CRITICAL_ALL,
        "recall_scorable_pct":
            100 * caught / N_CRITICAL_SCORABLE,
        "critical_precision_pct":
            100 * caught / n_flags,
        "flags_per_critical":
            n_flags / caught if caught else np.nan
    }

budgets = [0.05, 0.10, 0.20]
results = []

for budget in budgets:
    n_flags = int(N * budget)

    results.append(
        calculate_metrics(
            confidence_order,
            n_flags,
            "Raw confidence",
            budget
        )
    )

    results.append(
        calculate_metrics(
            hybrid_order,
            n_flags,
            "Disagreement + confidence",
            budget
        )
    )

# Binary disagreement has ties.
# Evaluate its stability using random tie-breaking.
dis_idx = np.where(dis == 1)[0]
agree_idx = np.where(dis == 0)[0]

rng = np.random.default_rng(42)
random_tie_results = []

for repeat in range(30):

    order = np.concatenate([
        rng.permutation(dis_idx),
        rng.permutation(agree_idx)
    ])

    for budget in budgets:
        n_flags = int(N * budget)

        random_tie_results.append(
            calculate_metrics(
                order,
                n_flags,
                "Binary disagreement (random ties)",
                budget
            )
        )

# Summarize random-tie runs
tie_df = pd.DataFrame(random_tie_results)

tie_summary = tie_df.groupby(
    "budget_pct", as_index=False
).agg(
    mean_caught=("critical_caught", "mean"),
    sd_caught=("critical_caught", "std"),
    mean_recall_all_pct=("recall_all_pct", "mean"),
    mean_recall_scorable_pct=("recall_scorable_pct", "mean")
)

results_df = pd.DataFrame(results)

print("\n===== FIXED-BUDGET RESULTS =====")
print(
    results_df.round(3).to_string(index=False)
)

print("\n===== BINARY DISAGREEMENT: RANDOM TIES =====")
print(
    tie_summary.round(3).to_string(index=False)
)

# Evaluate all disagreements without a fixed budget
all_disagreement_caught = int(
    critical[dis == 1].sum()
)

print("\n===== PURE DISAGREEMENT OPERATING POINT =====")
print(f"Flagging budget: {100 * dis.mean():.2f}%")
print("Total flags:", int(dis.sum()))
print("Critical errors caught:", all_disagreement_caught)
print(
    "Recall (all critical errors): "
    f"{100 * all_disagreement_caught / 266:.2f}%"
)
print(
    "Recall (scorable critical errors): "
    f"{100 * all_disagreement_caught / 245:.2f}%"
)

# Save
results_df.to_csv(
    RESULTS_DIR / "disagreement_budget_comparison.csv",
    index=False
)

tie_summary.to_csv(
    RESULTS_DIR / "disagreement_random_tie_sensitivity.csv",
    index=False
)

print("\nSTEP 9.2 COMPLETE!")
print("Comparison and sensitivity CSVs saved.")


In [ ]:

# ============================================================
# TASK 8 — REVIEW BURDEN ANALYSIS
# Fixed-budget flag counts and estimated review time
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

df = pd.read_csv(
    RESULTS_DIR / "fareez_cross_model_disagreement.csv",
    keep_default_na=False
)

confidence = pd.to_numeric(
    df["base_confidence"]
).to_numpy(dtype=float)

disagreement = pd.to_numeric(
    df["disagreement"]
).to_numpy(dtype=int)

critical = pd.to_numeric(
    df["base_is_critical_error"]
).to_numpy(dtype=int)

N = len(df)
ids = np.arange(N)
encounter_ids = sorted(df["encounter_id"].unique())

assert N == 352479
assert len(encounter_ids) == 272
assert critical.sum() == 245

# Identical ranking rules as Task 9.2
rankings = {
    "Raw confidence": np.lexsort(
        (ids, confidence)
    ),
    "Disagreement + confidence": np.lexsort(
        (ids, confidence, -disagreement)
    )
}

budgets = [0.05, 0.10, 0.20]
review_seconds = [5, 10, 20]
fixed_overhead_seconds = 30

summary_rows = []
encounter_rows = []

for method, ordering in rankings.items():

    for budget in budgets:

        n_flags = int(N * budget)

        flag_mask = np.zeros(N, dtype=bool)
        flag_mask[ordering[:n_flags]] = True

        caught_mask = flag_mask & (critical == 1)

        counts = pd.DataFrame({
            "encounter_id": df["encounter_id"],
            "flags": flag_mask.astype(int),
            "caught": caught_mask.astype(int)
        }).groupby("encounter_id").sum()

        counts = counts.reindex(
            encounter_ids, fill_value=0
        )

        flags = counts["flags"].to_numpy()
        caught = counts["caught"].to_numpy()

        total_flags = int(flags.sum())
        total_caught = int(caught.sum())

        for seconds in review_seconds:

            # No fixed review overhead for an encounter
            # with zero flags.
            minutes = (
                flags * seconds
                + (flags > 0) * fixed_overhead_seconds
            ) / 60

            summary_rows.append({
                "method": method,
                "budget_pct": budget * 100,
                "seconds_per_flag": seconds,
                "total_flags": total_flags,
                "critical_caught": total_caught,
                "flags_per_caught_error":
                    total_flags / total_caught
                    if total_caught else np.nan,
                "critical_recall_all_pct":
                    100 * total_caught / 266,
                "mean_flags_per_encounter": flags.mean(),
                "median_flags_per_encounter": np.median(flags),
                "mean_minutes_per_encounter": minutes.mean(),
                "median_minutes_per_encounter": np.median(minutes),
                "p90_minutes_per_encounter":
                    np.percentile(minutes, 90),
                "total_review_hours": minutes.sum() / 60
            })

            for i, eid in enumerate(encounter_ids):
                encounter_rows.append({
                    "encounter_id": eid,
                    "method": method,
                    "budget_pct": budget * 100,
                    "seconds_per_flag": seconds,
                    "flags": int(flags[i]),
                    "critical_caught": int(caught[i]),
                    "estimated_minutes": float(minutes[i])
                })

summary = pd.DataFrame(summary_rows)
encounter_detail = pd.DataFrame(encounter_rows)

# Save both datasets
summary.to_csv(
    RESULTS_DIR / "review_burden_summary.csv",
    index=False
)

encounter_detail.to_csv(
    RESULTS_DIR / "review_burden_per_encounter.csv",
    index=False
)

print("===== REVIEW BURDEN: 10% BUDGET =====")

view = summary[
    (summary["budget_pct"] == 10)
].copy()

print(
    view[[
        "method",
        "seconds_per_flag",
        "total_flags",
        "critical_caught",
        "flags_per_caught_error",
        "mean_minutes_per_encounter",
        "median_minutes_per_encounter",
        "p90_minutes_per_encounter"
    ]].round(2).to_string(index=False)
)

print("\n===== ALL REVIEW BUDGETS =====")

view = summary[
    summary["seconds_per_flag"] == 10
]

print(
    view[[
        "method",
        "budget_pct",
        "total_flags",
        "critical_caught",
        "flags_per_caught_error",
        "mean_minutes_per_encounter"
    ]].round(2).to_string(index=False)
)

print("\nSaved review burden summary and encounter details.")
print("TASK 8 ANALYSIS COMPLETE!")


In [ ]:

# ============================================================
# STEP 10.2 — LOCATE RISK-AWARE MODEL ARTIFACTS
# ============================================================

from pathlib import Path

REPO_DIR = Path("/content/Risk-aware-ASR")
PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")

roots = [
    REPO_DIR / "models",
    PROJECT_DIR
]

print("===== CORRECTED MODEL SEARCH =====")

found_models = []

for root in roots:
    if not root.exists():
        print("Not found:", root)
        continue

    for pattern in ["*.joblib", "*.pkl", "*.pickle"]:
        for p in root.rglob(pattern):
            found_models.append(p)
            print("Model:", p)
            print("Size:", round(p.stat().st_size / 1024, 2), "KB")

if not found_models:
    print("No model files found.")

print("\n===== CORRECTED MODEL CHECK =====")

corrected = [
    p for p in found_models
    if "no_tier" in p.name.lower()
]

if corrected:
    print("Corrected classifier found!")
    for p in corrected:
        print(p)
else:
    print("Corrected tier-free classifier not found.")
    print("May need the model artifact from your teammate.")

print("\n===== EXISTING RISK-SCORE DATA =====")

for root in [PROJECT_DIR / "fareez_followup_results",
             REPO_DIR / "data/processed"]:
    if root.exists():
        for p in root.glob("*.csv"):
            if "risk" in p.name.lower():
                print(p)

print("\nMODEL ARTIFACT CHECK COMPLETE!")


In [ ]:

# ============================================================
# TASK 9 — STEP 10.3
# Controlled head-to-head comparison
# RiskAware-ASR vs Cross-model Disagreement
# ============================================================

!pip -q install jellyfish joblib

import importlib.util
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

REPO_DIR = Path("/content/Risk-aware-ASR")
RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

MODEL_PATH = (
    REPO_DIR / "models/uncertainty_xgb_no_tier.joblib"
)

# Load original risk-scoring functions
script_path = REPO_DIR / "src/model/risk_score.py"

spec = importlib.util.spec_from_file_location(
    "riskaware_scoring", script_path
)
risk_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(risk_module)

# Load trained classifier (do not retrain)
model = joblib.load(MODEL_PATH)

print("===== MODEL VERIFICATION =====")
print("Model:", MODEL_PATH.name)

expected_features = ["confidence", "word_len"]

model_features = getattr(model, "feature_names_in_", None)

if model_features is None:
    model_features = model.get_booster().feature_names

print("Model features:", model_features)

assert model_features is not None
assert list(model_features) == expected_features, (
    "Unexpected model feature configuration."
)

assert model.n_features_in_ == 2

# Load glossary and original phonetic matching logic
glossary = risk_module.load_glossary() if False else None

glossary_df = pd.read_csv(
    REPO_DIR / "data/glossary/tiered_glossary.csv",
    keep_default_na=False
)

glossary = dict(zip(
    glossary_df["term"].str.lower(),
    glossary_df["tier"].astype(int)
))

phonetic_map = risk_module.build_phonetic_map(glossary)

# Load the existing shared evaluation dataset
df = pd.read_csv(
    RESULTS_DIR / "fareez_cross_model_disagreement.csv",
    keep_default_na=False
)

assert len(df) == 352479
assert df["encounter_id"].nunique() == 272

# Features derived ONLY from Whisper-base output
feature_input = pd.DataFrame({
    "whisper_word": df["base_word"].astype(str),
    "confidence": pd.to_numeric(df["base_confidence"])
})

features = risk_module.build_features(
    feature_input, glossary, phonetic_map
)

assert len(features) == len(df)
assert features.index.equals(df.index)

X = features[expected_features]

# Predict word-error probability
uncertainty = model.predict_proba(X)[:, 1]

# Apply existing clinical criticality weights
tier_weights = features["tier_whisper"].map(
    risk_module.TIER_WEIGHTS
).to_numpy(dtype=float)

risk_scores = uncertainty * tier_weights

assert np.isfinite(uncertainty).all()
assert np.isfinite(risk_scores).all()

# Attach outputs to evaluation dataframe
df["tier_whisper"] = features["tier_whisper"].to_numpy()
df["word_len"] = features["word_len"].to_numpy()
df["learned_uncertainty"] = uncertainty
df["tier_weight"] = tier_weights
df["risk_score"] = risk_scores

print("\n===== RISK SCORE SUMMARY =====")
print("Words scored:", len(df))
print("ASR-output tier distribution:")
print(df["tier_whisper"].value_counts().sort_index())

print("\nLearned uncertainty:")
print(df["learned_uncertainty"].describe().round(4))

print("\nRisk score:")
print(df["risk_score"].describe().round(4))

# Save without modifying the original files
output_path = RESULTS_DIR / "fareez_riskaware_scored.csv"

df.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("Risk-score generation complete!")


In [ ]:

# ============================================================
# FIXED-BUDGET EVALUATION — FOUR METHODS
# ============================================================

import numpy as np
import pandas as pd

N = len(df)
idx = np.arange(N)

confidence = pd.to_numeric(
    df["base_confidence"]
).to_numpy(dtype=float)

disagreement = pd.to_numeric(
    df["disagreement"]
).to_numpy(dtype=int)

critical = pd.to_numeric(
    df["base_is_critical_error"]
).to_numpy(dtype=int)

uncertainty = df["learned_uncertainty"].to_numpy()
risk = df["risk_score"].to_numpy()

assert critical.sum() == 245
assert disagreement.sum() == 28896

# Build deterministic rankings
rankings = {
    "Raw confidence": np.lexsort(
        (idx, confidence)
    ),

    "Learned uncertainty": np.lexsort(
        (idx, -uncertainty)
    ),

    "RiskAware-ASR": np.lexsort(
        (idx, -risk)
    ),

    "Disagreement + confidence": np.lexsort(
        (idx, confidence, -disagreement)
    )
}

rows = []

for method, order in rankings.items():

    for budget in [0.05, 0.10, 0.20]:

        n_flags = int(N * budget)
        selected = order[:n_flags]

        caught = int(critical[selected].sum())

        rows.append({
            "method": method,
            "budget_pct": int(budget * 100),
            "flagged_words": n_flags,
            "critical_caught": caught,
            "recall_all_pct": 100 * caught / 266,
            "recall_scorable_pct": 100 * caught / 245,
            "precision_pct": 100 * caught / n_flags,
            "flags_per_critical": (
                n_flags / caught if caught else np.nan
            )
        })

results = pd.DataFrame(rows)

print("===== FINAL HEAD-TO-HEAD COMPARISON =====")

print(
    results.round(3).to_string(index=False)
)

# Verify that raw-confidence ranking reproduces Step 9.2
raw_caught = results.loc[
    results["method"] == "Raw confidence",
    "critical_caught"
].tolist()

assert raw_caught == [71, 129, 186], (
    "Raw-confidence baseline does not match Step 9.2."
)

# Save comparison
results.to_csv(
    RESULTS_DIR / "final_four_method_comparison.csv",
    index=False
)

# Show primary 10% comparison
print("\n===== PRIMARY 10% REVIEW BUDGET =====")

primary = results[
    results["budget_pct"] == 10
].sort_values(
    "recall_all_pct", ascending=False
)

print(
    primary[[
        "method",
        "critical_caught",
        "recall_all_pct",
        "precision_pct",
        "flags_per_critical"
    ]].round(3).to_string(index=False)
)

print("\nFinal comparison saved successfully!")


In [ ]:

# ============================================================
# TASK 9 — STEP 10.4
# PAIRED BOOTSTRAP: RISK-AWARE VS CROSS-MODEL DISAGREEMENT
# Exact review budget within each resample
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

# 1. Load scored Whisper-base words
df = pd.read_csv(
    RESULTS_DIR / "fareez_riskaware_scored.csv",
    keep_default_na=False,
    usecols=[
        "encounter_id",
        "base_confidence",
        "disagreement",
        "base_is_critical_error",
        "learned_uncertainty",
        "risk_score"
    ]
)

# 2. Load reference-based labels
labels = pd.read_csv(
    RESULTS_DIR / "fareez_base_word_labels.csv",
    keep_default_na=False,
    usecols=[
        "encounter_id", "gold_word", "tier", "is_error"
    ]
)

# Exclude insertion rows with no reference word
gold = labels[
    labels["gold_word"].astype(str).str.strip().ne("")
].copy()

gold["tier"] = pd.to_numeric(gold["tier"])
gold["is_error"] = pd.to_numeric(gold["is_error"])

gold["critical_error"] = (
    (gold["tier"] >= 2)
    & (gold["is_error"] == 1)
).astype(int)

encounters = sorted(df["encounter_id"].unique())
E = len(encounters)
N = len(df)

assert E == 272
assert N == 352479
assert gold["critical_error"].sum() == 266

# 3. Encode encounter IDs
encounter_index = {
    eid: i for i, eid in enumerate(encounters)
}

enc_idx = df["encounter_id"].map(
    encounter_index
).to_numpy(dtype=np.int32)

# Reference critical errors per encounter
reference_critical = (
    gold.groupby("encounter_id")["critical_error"]
    .sum()
    .reindex(encounters, fill_value=0)
    .to_numpy(dtype=np.int64)
)

words_per_encounter = np.bincount(
    enc_idx, minlength=E
)

critical = pd.to_numeric(
    df["base_is_critical_error"]
).to_numpy(dtype=np.int32)

assert critical.sum() == 245
assert reference_critical.sum() == 266

# 4. Build four deterministic rankings
idx = np.arange(N)

conf = pd.to_numeric(
    df["base_confidence"]
).to_numpy(dtype=float)

dis = pd.to_numeric(
    df["disagreement"]
).to_numpy(dtype=int)

uncertainty = pd.to_numeric(
    df["learned_uncertainty"]
).to_numpy(dtype=float)

risk = pd.to_numeric(
    df["risk_score"]
).to_numpy(dtype=float)

rankings = {
    "Raw confidence":
        np.lexsort((idx, conf)),

    "Learned uncertainty":
        np.lexsort((idx, -uncertainty)),

    "RiskAware-ASR":
        np.lexsort((idx, -risk)),

    "Disagreement + confidence":
        np.lexsort((idx, conf, -dis))
}

budgets = [5, 10, 20]

# 5. Audit the original 10% results
expected_10 = {
    "Raw confidence": 129,
    "Learned uncertainty": 134,
    "RiskAware-ASR": 155,
    "Disagreement + confidence": 209
}

for method, order in rankings.items():
    caught = int(
        critical[order[:int(N * 0.10)]].sum()
    )
    assert caught == expected_10[method], (
        f"Baseline mismatch for {method}"
    )

print("Original 10% results reproduced.")

# Precompute encounter IDs and labels in ranking order
ranked_data = {
    method: (enc_idx[order], critical[order])
    for method, order in rankings.items()
}

# 6. Bootstrap
N_BOOT = 2000
rng = np.random.default_rng(42)

boot_recall = {
    method: {
        budget: np.zeros(N_BOOT)
        for budget in budgets
    }
    for method in rankings
}

print("Starting paired encounter bootstrap...")

for iteration in range(N_BOOT):

    # Sample encounters with replacement
    sampled = rng.integers(0, E, size=E)

    # How often each encounter occurs
    weights = np.bincount(
        sampled, minlength=E
    ).astype(np.int64)

    # Weighted total words and critical errors
    total_words = int(
        np.dot(weights, words_per_encounter)
    )

    total_critical = int(
        np.dot(weights, reference_critical)
    )

    if total_critical == 0:
        continue

    for method, (rank_enc, rank_critical) in ranked_data.items():

        # Each word inherits its encounter's resample weight
        word_weights = weights[rank_enc]

        cumulative_words = np.cumsum(
            word_weights, dtype=np.int64
        )

        cumulative_critical = np.cumsum(
            word_weights * rank_critical,
            dtype=np.int64
        )

        for budget in budgets:

            k = int(total_words * budget / 100)

            # Locate the exact budget boundary
            pos = int(
                np.searchsorted(
                    cumulative_words, k, side="left"
                )
            )

            before_words = (
                cumulative_words[pos - 1]
                if pos > 0 else 0
            )

            before_caught = (
                cumulative_critical[pos - 1]
                if pos > 0 else 0
            )

            # Handle a boundary inside duplicated encounters
            boundary_count = k - before_words

            caught = (
                before_caught
                + boundary_count * rank_critical[pos]
            )

            boot_recall[method][budget][iteration] = (
                100 * caught / total_critical
            )

    if (iteration + 1) % 500 == 0:
        print(
            f"Completed {iteration + 1}/{N_BOOT} resamples",
            flush=True
        )

# 7. Paired confidence intervals
rows = []

for budget in budgets:

    hybrid = boot_recall[
        "Disagreement + confidence"
    ][budget]

    riskaware = boot_recall[
        "RiskAware-ASR"
    ][budget]

    confidence = boot_recall[
        "Raw confidence"
    ][budget]

    for comparator, other in [
        ("RiskAware-ASR", riskaware),
        ("Raw confidence", confidence)
    ]:

        difference = hybrid - other

        ci_low, ci_high = np.percentile(
            difference, [2.5, 97.5]
        )

        # Original sample difference
        k_original = int(N * budget / 100)

        hybrid_caught = int(
            critical[
                rankings["Disagreement + confidence"][
                    :k_original
                ]
            ].sum()
        )

        other_caught = int(
            critical[
                rankings[comparator][:k_original]
            ].sum()
        )

        observed_gap = (
            100 * (hybrid_caught - other_caught) / 266
        )

        rows.append({
            "budget_pct": budget,
            "comparison":
                "Disagreement + confidence vs " + comparator,
            "observed_gap_pp": observed_gap,
            "ci_lower_pp": ci_low,
            "ci_upper_pp": ci_high,
            "bootstrap_samples": N_BOOT
        })

summary = pd.DataFrame(rows)

print("\n===== PAIRED BOOTSTRAP RESULTS =====")
print(summary.round(3).to_string(index=False))

# 8. Save results
output = (
    RESULTS_DIR / "final_method_comparison_bootstrap.csv"
)

summary.to_csv(output, index=False)

print("\nSaved:", output)
print("STEP 10.4 COMPLETE!")


In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/riskaware-asr")

BASE_DIR = PROJECT_DIR / "fareez_whisper_out"
SMALL_DIR = PROJECT_DIR / "fareez_whisper_small_auto"

print("===== FOLDER CHECK =====")
print("Base folder exists:", BASE_DIR.exists())
print("Small folder exists:", SMALL_DIR.exists())

base_files = sorted(BASE_DIR.rglob("*.json"))
small_files = sorted(SMALL_DIR.rglob("*.json"))

print("\n===== FILE COUNTS =====")
print("Base JSON files:", len(base_files))
print("Small JSON files:", len(small_files))

print("\n===== SAMPLE FILENAMES =====")
print("Base:", [p.name for p in base_files[:5]])
print("Small:", [p.name for p in small_files[:5]])

base_ids = {p.stem for p in base_files if p.stem.startswith("CAR")}
small_ids = {p.stem for p in small_files if p.stem.startswith("CAR")}

print("\n===== ENCOUNTER MATCHING =====")
print("Base encounter IDs:", len(base_ids))
print("Small encounter IDs:", len(small_ids))
print("Matching IDs:", len(base_ids & small_ids))
print("Missing from small:", sorted(base_ids - small_ids)[:15])
print("Missing from base:", sorted(small_ids - base_ids)[:15])

print("\n===== PROJECT FOLDERS =====")
for p in sorted(PROJECT_DIR.iterdir()):
    if p.is_dir():
        print(p.name)


In [ ]:
# STEP 1 — Setup and connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, math, time, re
from pathlib import Path

subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y ffmpeg >/dev/null'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai-whisper', 'jiwer', 'pandas', 'numpy', 'jellyfish', 'xgboost', 'scikit-learn', 'joblib'], check=True)
REPO = Path('/content/Risk-aware-ASR')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/mih1ka/Risk-aware-ASR.git',str(REPO)],check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# EDIT THESE TWO PATHS to point at your actual Fareez data.
DRIVE_ROOT = Path('/content/drive/MyDrive/riskaware-asr')
AUDIO_DIR  = DRIVE_ROOT / 'fareez_audio'  # User-supplied real Fareez audio
BASE_DIR   = DRIVE_ROOT / 'whisper_out'   # Existing Whisper-base JSONs for Fareez

SMALL_DIR = DRIVE_ROOT / 'whisper_small_fareez'
RESULT_DIR = DRIVE_ROOT / 'fareez_followup_results'
SMALL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Audio input:', AUDIO_DIR)
print('Base output:', BASE_DIR)
print('Small output:', SMALL_DIR)

## Reporting checklist

- **Task 2:** document matched encounter count, Whisper version/configuration, device, pooled and macro WER, tier-specific error rates, differences from base, and representative cases.
- **Task 9:** describe disagreement alignment, how substitutions/deletions/insertions were treated, tie-breaking for fixed flag budgets, and performance against both gold and single-model confidence/risk baselines. A disagreement signal does not independently mean either model is wrong.
- **Task 8:** report absolute flags, *flags per caught true critical error*, *flags per all true critical errors*, critical recall (with unscorable deletions documented), per-encounter mean/median, and a sensitivity range of hypothetical review times; do not call estimated times measured times.
- **Tasks 6–7:** blind rating protocol/rubric, 30-item stratification, rater qualifications, independent ratings, agreement and κ (and limitations). Do not invent raters or scores.
- **Paper:** introduction and related work can be drafted now; methodology should distinguish synthetic ACI-Bench TTS, real Fareez audio, and production-ASR comparisons, and use the finalized normalization and glossary version. Treat the current findings in `report/decisions_log.md` as historical until rerun/verified.

**Reproducibility:** Freeze your Git commit hash, note the Colab/Whisper/Python versions and date, preserve output hashes, and retain input lists. Do not overwrite previously published 272-encounter reference results with a smaller pilot run.

In [ ]:

from pathlib import Path
import subprocess

FORK_URL = "https://github.com/yea-shvii/Risk-aware-ASR.git"
FORK_DIR = Path("/content/Risk-aware-ASR-fork")

if not (FORK_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", FORK_URL, str(FORK_DIR)],
        check=True
    )

print("\n===== REPOSITORY =====")
subprocess.run(
    ["git", "-C", str(FORK_DIR), "remote", "-v"],
    check=True
)

print("\nClone successful!")


In [ ]:

BRANCH = "fareez-followup-oct-2026"

subprocess.run([
    "git", "-C", str(FORK_DIR),
    "switch", "-c", BRANCH
], check=True)

print("Working branch:", BRANCH)


In [ ]:

# ============================================================
# FIND SAVED RISK-AWARE ASR EXPERIMENT RESULTS
# ============================================================

from pathlib import Path
from collections import Counter

DRIVE = Path("/content/drive/MyDrive")
PROJECT = DRIVE / "riskaware-asr"
EXPECTED = PROJECT / "fareez_followup_results"

print("===== DRIVE STATUS =====")
print("MyDrive accessible:", DRIVE.exists())
print("Project folder exists:", PROJECT.exists())
print("Expected results folder exists:", EXPECTED.exists())

print("\n===== PROJECT FOLDERS =====")
if PROJECT.exists():
    for p in sorted(PROJECT.iterdir()):
        print(
            "DIR " if p.is_dir() else "FILE",
            p.name
        )

print("\n===== EXPECTED RESULTS FOLDER =====")
if EXPECTED.exists():
    for p in sorted(EXPECTED.iterdir()):
        if p.is_file():
            print(p.name,
                  f"({p.stat().st_size / 1024:.1f} KB)")
else:
    print("Expected results folder not found.")

print("\n===== SEARCHING FOR CSV RESULTS =====")

# Search within the project rather than guessing paths
csv_files = (
    sorted(PROJECT.rglob("*.csv"))
    if PROJECT.exists() else []
)

print("Total CSV files found:", len(csv_files))

for p in csv_files:
    print(p.relative_to(PROJECT))

print("\n===== IMPORTANT FILE CHECK =====")

needed = [
    "base_vs_small_tier_comparison.csv",
    "tier_bootstrap_95ci.csv",
    "disagreement_budget_comparison.csv",
    "review_burden_summary.csv",
    "final_four_method_comparison.csv",
    "final_method_comparison_bootstrap.csv"
]

for name in needed:
    matches = [
        p for p in csv_files if p.name == name
    ]

    if matches:
        for p in matches:
            print("FOUND:", p)
    else:
        print("MISSING:", name)

print("\nDIAGNOSTIC COMPLETE!")


In [ ]:

# ============================================================
# STEP 4A — RECONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Define paths
DRIVE = Path("/content/drive/MyDrive")
PROJECT = DRIVE / "riskaware-asr"
RESULTS_DIR = PROJECT / "fareez_followup_results"

print("\n===== DRIVE VERIFICATION =====")
print("MyDrive accessible:", DRIVE.exists())
print("Project folder exists:", PROJECT.exists())
print("Results folder exists:", RESULTS_DIR.exists())

print("\n===== SAVED EXPERIMENT RESULTS =====")

if RESULTS_DIR.exists():
    csv_files = sorted(RESULTS_DIR.glob("*.csv"))

    print("Total CSV files:", len(csv_files))

    for file in csv_files:
        print(
            file.name,
            f"({file.stat().st_size / 1024:.1f} KB)"
        )
else:
    print("Results folder not accessible.")

print("\n===== GITHUB CLONE CHECK =====")
FORK_DIR = Path("/content/Risk-aware-ASR-fork")
print("Fork clone exists:", (FORK_DIR / ".git").exists())


In [ ]:

# ============================================================
# STEP 4B — COPY VERIFIED EXPERIMENT RESULTS
# ============================================================

from pathlib import Path
import shutil
import subprocess

FORK_DIR = Path("/content/Risk-aware-ASR-fork")

SOURCE = Path(
    "/content/drive/MyDrive/riskaware-asr/fareez_followup_results"
)

DEST = FORK_DIR / "experiments/fareez_followup/results"
DEST.mkdir(parents=True, exist_ok=True)

# Verify this clone points to your fork
origin = subprocess.check_output(
    ["git", "-C", str(FORK_DIR),
     "remote", "get-url", "origin"],
    text=True
).strip()

print("Git origin:", origin)

assert "yea-shvii/risk-aware-asr" in origin.lower(), (
    "Git origin does not point to your fork!"
)

# Experiment results to upload
files_to_push = [
    "pilot_base_vs_small_wer.csv",
    "fareez_base_encounter_wer.csv",
    "fareez_small_encounter_wer.csv",
    "base_vs_small_tier_comparison.csv",
    "tier_bootstrap_95ci.csv",
    "disagreement_budget_comparison.csv",
    "disagreement_random_tie_sensitivity.csv",
    "review_burden_summary.csv",
    "final_four_method_comparison.csv",
    "final_method_comparison_bootstrap.csv"
]

print("\n===== COPYING RESULTS =====")

for filename in files_to_push:
    src = SOURCE / filename

    assert src.is_file(), f"Missing: {filename}"
    assert src.stat().st_size < 5_000_000, (
        f"Unexpectedly large result: {filename}"
    )

    shutil.copy2(src, DEST / filename)
    print("Copied:", filename)

print("\n===== FINAL VERIFICATION =====")

copied = sorted(DEST.glob("*.csv"))

print("CSV files ready:", len(copied))
print("Total size:",
      round(sum(p.stat().st_size for p in copied) / 1024, 2),
      "KB")

assert len(copied) == 10

print("\nGit branch:")
subprocess.run([
    "git", "-C", str(FORK_DIR),
    "branch", "--show-current"
], check=True)

print("\nRESULTS READY FOR GITHUB!")


In [ ]:

# ============================================================
# FIND YOUR COLAB NOTEBOOK
# ============================================================

from pathlib import Path

NOTEBOOK_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks"
)

notebooks = sorted(NOTEBOOK_DIR.glob("*.ipynb"))

print("===== AVAILABLE NOTEBOOKS =====")

for i, notebook in enumerate(notebooks, 1):
    print(f"{i}. {notebook.name}")

print("\nTotal notebooks:", len(notebooks))
